# Hop log-prob analysis

This notebook compares the hop-by-hop `eval-owl` outputs for `filtered-dataset-lora-8-seed-42` and `filtered-dataset-dpoints-only-lora-8-seed-42` across `hop1_noprompt` through `hop9_noprompt`.

The notebook is designed to do three things: export reusable CSVs, generate the requested figures, and save every figure into this folder's `images/` directory for later reuse.

In [1]:
from __future__ import annotations

import json
import math
import re
from pathlib import Path

import matplotlib
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "figure.dpi": 120,
    "savefig.dpi": 220,
})


In [2]:
def find_project_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "workspace").exists():
            return candidate
    return start

In [3]:



PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks" / "hop_logprob_analysis"
CSV_DIR = NOTEBOOK_DIR / "csv"
IMAGE_DIR = NOTEBOOK_DIR / "images"
DATA_ROOT = PROJECT_ROOT / "workspace" / "multihop" / "qwen" / "owl"
HOPS = list(range(1, 10))

RUNS = {
    "lora": {
        "label": "filtered-dataset-lora-8-seed-42",
        "short_label": "lora-8",
        "dataset_dir": "filtered-dataset-lora-8-seed-42",
    },
    "dpoints_only": {
        "label": "filtered-dataset-dpoints-only-lora-8-seed-42",
        "short_label": "dpoints-only",
        "dataset_dir": "filtered-dataset-dpoints-only-lora-8-seed-42",
    },
}

In [4]:


for directory in (CSV_DIR, IMAGE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

In [5]:


def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

def build_run_dir(hop: int, run_key: str) -> Path:
    return DATA_ROOT / f"hop{hop}_noprompt" / "seed-42" / RUNS[run_key]["dataset_dir"]

def find_latest_checkpoint(eval_owl_dir: Path) -> Path:
    matches = []
    for child in eval_owl_dir.iterdir():
        match = re.fullmatch(r"checkpoint-(\d+)", child.name)
        if child.is_dir() and match:
            matches.append((int(match.group(1)), child))
    if not matches:
        raise FileNotFoundError(f"No checkpoint-* folders found in {eval_owl_dir}")
    return max(matches, key=lambda item: item[0])[1]

def load_logprob_stats(run_dir: Path, checkpoint_group: str) -> tuple[dict, str, Path]:
    eval_owl_dir = run_dir / "eval-owl"
    if checkpoint_group == "base":
        stats_path = eval_owl_dir / "base" / "logprob_stats.json"
        checkpoint_name = "base"
    elif checkpoint_group == "final":
        checkpoint_dir = find_latest_checkpoint(eval_owl_dir)
        stats_path = checkpoint_dir / "logprob_stats.json"
        checkpoint_name = checkpoint_dir.name
    else:
        raise ValueError(f"Unsupported checkpoint group: {checkpoint_group}")
    if not stats_path.exists():
        raise FileNotFoundError(f"Missing logprob stats file: {stats_path}")
    return load_json(stats_path), checkpoint_name, stats_path

def summarise_stats(stats: dict, hop: int, run_key: str, checkpoint_group: str, checkpoint_name: str) -> dict:
    per_question = stats["per_question"]
    log_probs = [float(row["log_p_target"]) for row in per_question]
    probs = [math.exp(value) for value in log_probs]
    mean_log_p = float(stats["mean_log_p_target"])
    return {
        "hop": hop,
        "run_key": run_key,
        "run_label": RUNS[run_key]["short_label"],
        "dataset_dir": RUNS[run_key]["label"],
        "checkpoint_group": checkpoint_group,
        "checkpoint_name": checkpoint_name,
        "n_questions": len(per_question),
        "mean_log_p_target": mean_log_p,
        "mean_p_target": sum(probs) / len(probs),
        "exp_mean_log_p_target": math.exp(mean_log_p),
        "min_log_p_target": min(log_probs),
        "max_log_p_target": max(log_probs),
    }

def save_figure(fig: plt.Figure, filename: str) -> Path:
    path = IMAGE_DIR / filename
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path

## Data layout and exports

The code below reads the base and latest checkpoint for each hop/run pair, then writes CSVs that are useful for replotting later without rerunning the notebook.

I keep both a long table and a comparison table because the long table is better for faceting and the wide table is easier to inspect in a spreadsheet.

In [6]:
def flatten_columns(columns) -> list[str]:
    flat_columns = []
    for column in columns:
        if isinstance(column, tuple):
            pieces = [str(piece) for piece in column if piece not in ("", None)]
            flat_columns.append("_".join(pieces))
        else:
            flat_columns.append(str(column))
    return flat_columns

summary_rows = []
question_rows = []

for hop in HOPS:
    for run_key in RUNS:
        run_dir = build_run_dir(hop, run_key)
        for checkpoint_group in ("base", "final"):
            stats, checkpoint_name, stats_path = load_logprob_stats(run_dir, checkpoint_group)
            summary_row = summarise_stats(stats, hop, run_key, checkpoint_group, checkpoint_name)
            summary_row["stats_path"] = str(stats_path)
            summary_rows.append(summary_row)

            for question_idx, question_row in enumerate(stats["per_question"], start=1):
                log_p_target = float(question_row["log_p_target"])
                question_rows.append({
                    "hop": hop,
                    "run_key": run_key,
                    "run_label": RUNS[run_key]["short_label"],
                    "dataset_dir": RUNS[run_key]["label"],
                    "checkpoint_group": checkpoint_group,
                    "checkpoint_name": checkpoint_name,
                    "question_idx": question_idx,
                    "question": question_row["question"],
                    "log_p_target": log_p_target,
                    "p_target": math.exp(log_p_target),
                })

summary_long_df = pd.DataFrame(summary_rows).sort_values(["hop", "run_key", "checkpoint_group"]).reset_index(drop=True)
question_long_df = pd.DataFrame(question_rows).sort_values(["hop", "run_key", "checkpoint_group", "question_idx"]).reset_index(drop=True)

base_summary_df = summary_long_df[summary_long_df["checkpoint_group"] == "base"].copy()
final_summary_df = summary_long_df[summary_long_df["checkpoint_group"] == "final"].copy()

summary_compare_df = base_summary_df.merge(
    final_summary_df,
    on=["hop", "run_key", "run_label", "dataset_dir"],
    suffixes=("_base", "_final"),
)
summary_compare_df["delta_mean_log_p_target"] = summary_compare_df["mean_log_p_target_final"] - summary_compare_df["mean_log_p_target_base"]
summary_compare_df["delta_mean_p_target"] = summary_compare_df["mean_p_target_final"] - summary_compare_df["mean_p_target_base"]
summary_compare_df["ratio_mean_p_target"] = summary_compare_df["mean_p_target_final"] / summary_compare_df["mean_p_target_base"]
summary_compare_df["percent_change_mean_p_target"] = (summary_compare_df["ratio_mean_p_target"] - 1.0) * 100.0

base_questions_df = question_long_df[question_long_df["checkpoint_group"] == "base"].copy()
final_questions_df = question_long_df[question_long_df["checkpoint_group"] == "final"].copy()
question_delta_df = base_questions_df.merge(
    final_questions_df,
    on=["hop", "run_key", "run_label", "dataset_dir", "question_idx", "question"],
    suffixes=("_base", "_final"),
)
question_delta_df["delta_log_p_target"] = question_delta_df["log_p_target_final"] - question_delta_df["log_p_target_base"]
question_delta_df["delta_p_target"] = question_delta_df["p_target_final"] - question_delta_df["p_target_base"]

final_comparison_questions = final_questions_df.pivot_table(
    index=["hop", "question_idx", "question"],
    columns="run_key",
    values=["log_p_target", "p_target"],
    aggfunc="first",
).reset_index()
final_comparison_questions.columns = flatten_columns(final_comparison_questions.columns)
final_comparison_questions["delta_log_p_target_dpoints_only_minus_lora"] = final_comparison_questions["log_p_target_dpoints_only"] - final_comparison_questions["log_p_target_lora"]
final_comparison_questions["delta_p_target_dpoints_only_minus_lora"] = final_comparison_questions["p_target_dpoints_only"] - final_comparison_questions["p_target_lora"]

summary_long_path = CSV_DIR / "hop_checkpoint_summary_long.csv"
summary_compare_path = CSV_DIR / "hop_checkpoint_summary_wide.csv"
question_long_path = CSV_DIR / "question_logprob_long.csv"
question_delta_path = CSV_DIR / "question_delta_within_run_long.csv"
question_final_path = CSV_DIR / "question_final_comparison_wide.csv"

summary_long_df.to_csv(summary_long_path, index=False)
summary_compare_df.to_csv(summary_compare_path, index=False)
question_long_df.to_csv(question_long_path, index=False)
question_delta_df.to_csv(question_delta_path, index=False)
final_comparison_questions.to_csv(question_final_path, index=False)

summary_long_df[["hop", "run_label", "checkpoint_group", "checkpoint_name", "mean_log_p_target", "mean_p_target"]].head(6)

,hop,run_label,checkpoint_group,checkpoint_name,mean_log_p_target,mean_p_target
0,1,dpoints-only,base,base,-12.383125,0.000047
1,1,dpoints-only,final,checkpoint-552,-12.580000,0.000040
2,1,lora-8,base,base,-12.383125,0.000047
3,1,lora-8,final,checkpoint-668,-12.193125,0.000054
4,2,dpoints-only,base,base,-12.383125,0.000047
5,2,dpoints-only,final,checkpoint-552,-12.486250,0.000047


## Main comparison plot

This is the requested view: the latest checkpoint for each run, shown across hops 1 through 11. I use separate panels for log-probability and probability because the scales are different and a secondary axis would make the comparison harder to read.

The probability panel uses a log y-axis because the values are tiny and the trend is easier to see that way.

In [7]:
final_only_df = final_summary_df.copy()
checkpoint_labels = {
    run_key: final_only_df.loc[final_only_df["run_key"] == run_key, "checkpoint_name"].iloc[0]
    for run_key in RUNS
}

fig, axes = plt.subplots(2, 1, figsize=(11, 9), sharex=True)
for run_key, group in final_only_df.groupby("run_key"):
    axes[0].plot(group["hop"], group["mean_log_p_target"], marker="o", linewidth=2, label=f"{RUNS[run_key]['short_label']} ({checkpoint_labels[run_key]})")
    axes[1].plot(group["hop"], group["mean_p_target"], marker="o", linewidth=2, label=f"{RUNS[run_key]['short_label']} ({checkpoint_labels[run_key]})")

axes[0].set_title("Final checkpoint mean log p(owl) by hop")
axes[0].set_ylabel("Mean log p(owl)")
axes[0].legend(loc="best")

axes[1].set_title("Final checkpoint mean p(owl) by hop")
axes[1].set_ylabel("Mean p(owl) [log scale]")
axes[1].set_xlabel("Hop")
axes[1].set_yscale("log")
axes[1].legend(loc="best")
axes[1].set_xticks(HOPS)

main_plot_path = save_figure(fig, "final_checkpoint_mean_logprob_and_p_by_hop.png")
main_plot_path

PosixPath('/home/abasso_aims_ac_za/divergence-tokens/notebooks/hop_logprob_analysis/images/final_checkpoint_mean_logprob_and_p_by_hop.png')

## Gap plots and within-run change

The cross-run gap plot is the cleanest way to see whether the dpoints-only run is above or below the lora run at each hop. The within-run change plot is still useful because it shows whether the final checkpoint improved consistently relative to the base model inside each run.

In [8]:
final_gap_df = final_only_df.pivot_table(
    index="hop",
    columns="run_key",
    values=["mean_log_p_target", "mean_p_target"],
    aggfunc="first",
).reset_index()
final_gap_df.columns = flatten_columns(final_gap_df.columns)
final_gap_df["delta_mean_log_p_target_dpoints_only_minus_lora"] = final_gap_df["mean_log_p_target_dpoints_only"] - final_gap_df["mean_log_p_target_lora"]
final_gap_df["delta_mean_p_target_dpoints_only_minus_lora"] = final_gap_df["mean_p_target_dpoints_only"] - final_gap_df["mean_p_target_lora"]

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
axes[0].axhline(0, color="black", linewidth=1)
axes[0].plot(final_gap_df["hop"], final_gap_df["delta_mean_log_p_target_dpoints_only_minus_lora"], marker="o", linewidth=2, color="#2a9d8f")
axes[0].set_title("Final checkpoint gap: dpoints-only minus lora")
axes[0].set_ylabel("Delta mean log p(owl)")

axes[1].axhline(0, color="black", linewidth=1)
axes[1].plot(final_gap_df["hop"], final_gap_df["delta_mean_p_target_dpoints_only_minus_lora"], marker="o", linewidth=2, color="#e76f51")
axes[1].set_title("Final checkpoint gap in probability: dpoints-only minus lora")
axes[1].set_ylabel("Delta mean p(owl)")
axes[1].set_xlabel("Hop")
axes[1].set_xticks(HOPS)
axes[1].set_yscale("symlog", linthresh=1e-7)

gap_plot_path = save_figure(fig, "final_checkpoint_gap_by_hop.png")
gap_plot_path

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
for run_key, group in summary_compare_df.groupby("run_key"):
    axes[0].plot(group["hop"], group["delta_mean_log_p_target"], marker="o", linewidth=2, label=RUNS[run_key]["short_label"])
    axes[1].plot(group["hop"], group["delta_mean_p_target"], marker="o", linewidth=2, label=RUNS[run_key]["short_label"])

axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("Within-run change in mean log p(owl): final minus base")
axes[0].set_ylabel("Delta mean log p(owl)")
axes[0].legend(loc="best")

axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("Within-run change in mean p(owl): final minus base")
axes[1].set_ylabel("Delta mean p(owl)")
axes[1].set_xlabel("Hop")
axes[1].set_xticks(HOPS)
axes[1].set_yscale("symlog", linthresh=1e-7)
axes[1].legend(loc="best")

training_effect_plot_path = save_figure(fig, "within_run_final_minus_base_change_by_hop.png")
training_effect_plot_path

PosixPath('/home/abasso_aims_ac_za/divergence-tokens/notebooks/hop_logprob_analysis/images/within_run_final_minus_base_change_by_hop.png')

## Per-question diagnostics and other plot ideas

The per-question plots answer a different question from the hop averages: are the gains broad-based, or are they driven by a few outlier prompts?

Other plot types I considered for later use are paired scatter plots, violin plots, and small-multiple grids. I kept the notebook focused on the plots below because they are the easiest to compare across 11 hops without becoming visually crowded.

In [9]:
# Heatmap of within-run training effect by question and hop.
fig, axes = plt.subplots(1, 2, figsize=(16, 12), sharey=True)
for ax, run_key in zip(axes, RUNS):
    run_delta = question_delta_df[question_delta_df["run_key"] == run_key]
    matrix = run_delta.pivot(index="question_idx", columns="hop", values="delta_log_p_target").sort_index()
    im = ax.imshow(matrix.values, aspect="auto", interpolation="nearest", cmap="coolwarm")
    ax.set_title(f"{RUNS[run_key]['short_label']}: final minus base log p(owl)")
    ax.set_xlabel("Hop")
    ax.set_xticks(range(len(matrix.columns)))
    ax.set_xticklabels([str(hop) for hop in matrix.columns])
    ax.set_ylabel("Question index")
    ax.set_yticks(range(len(matrix.index)))
    ax.set_yticklabels([str(idx) for idx in matrix.index])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

heatmap_path = save_figure(fig, "question_delta_heatmap_by_run.png")
heatmap_path

# Distribution view of the same values to spot hop-level spread and outliers.
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
for ax, run_key in zip(axes, RUNS):
    run_delta = question_delta_df[question_delta_df["run_key"] == run_key]
    data = [run_delta.loc[run_delta["hop"] == hop, "delta_log_p_target"].tolist() for hop in HOPS]
    ax.boxplot(data, labels=[str(hop) for hop in HOPS], showmeans=True)
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title(f"{RUNS[run_key]['short_label']}: per-question delta log p(owl)")
    ax.set_xlabel("Hop")
    ax.set_ylabel("Delta log p(owl)")
    ax.tick_params(axis="x", rotation=0)

boxplot_path = save_figure(fig, "question_delta_boxplot_by_run.png")
boxplot_path

# Paired scatter for the final checkpoints, colored by hop, to show direct run-vs-run agreement.
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
scenarios = [("log_p_target_dpoints_only", "log_p_target_lora", "Final log p(owl)"), ("p_target_dpoints_only", "p_target_lora", "Final p(owl)")]
for ax, (x_col, y_col, title) in zip(axes, scenarios):
    x = final_comparison_questions[x_col]
    y = final_comparison_questions[y_col]
    scatter = ax.scatter(x, y, c=final_comparison_questions["hop"], cmap="viridis", alpha=0.8, s=28)
    min_val = min(x.min(), y.min())
    max_val = max(x.max(), y.max())
    ax.plot([min_val, max_val], [min_val, max_val], color="black", linestyle="--", linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("dpoints-only")
    ax.set_ylabel("lora-8")
    fig.colorbar(scatter, ax=ax, label="Hop")

scatter_path = save_figure(fig, "paired_final_checkpoint_scatter_by_question.png")
scatter_path

/tmp/ipykernel_43822/1436294366.py:24: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=[str(hop) for hop in HOPS], showmeans=True)
/tmp/ipykernel_43822/1436294366.py:24: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=[str(hop) for hop in HOPS], showmeans=True)


PosixPath('/home/abasso_aims_ac_za/divergence-tokens/notebooks/hop_logprob_analysis/images/paired_final_checkpoint_scatter_by_question.png')

## Additional metrics from stats.json

Some runs include a `stats.json` (or other JSON) in the `eval-owl` folders that contain useful scalar metrics such as `p_owl`. The cells below search for those files across hops and runs, extract top-level numeric values (and simple nested numeric fields), write a CSV summary for easy re-use, and produce plots for `p_owl` plus other numeric metrics discovered.

Notes: the code is defensive and will skip missing files; it only pulls scalar numeric fields.

In [10]:
# Collect scalar numeric metrics from stats.json files (base and final) across hops and runs
metrics_rows = []

def try_stats_path_for_group(eval_owl_dir: Path, checkpoint_group: str) -> Path | None:
    # prefer named stats.json under base/ or checkpoint dir; return first existing path or None
    if checkpoint_group == 'base':
        candidate = eval_owl_dir / 'base' / 'stats.json'
        if candidate.exists():
            return candidate
        # some runs may use other names
        candidate2 = eval_owl_dir / 'base' / 'stats' / 'stats.json'
        if candidate2.exists():
            return candidate2
        return None
    else:
        # final: look for latest checkpoint and stats.json inside it
        try:
            ckpt = find_latest_checkpoint(eval_owl_dir)
        except Exception:
            return None
        candidate = ckpt / 'stats.json'
        if candidate.exists():
            return candidate
        # fallback inside a stats/ subfolder
        candidate2 = ckpt / 'stats' / 'stats.json'
        if candidate2.exists():
            return candidate2
        return None

for hop in HOPS:
    for run_key in RUNS:
        run_dir = build_run_dir(hop, run_key)
        eval_owl_dir = run_dir / 'eval-owl'
        if not eval_owl_dir.exists():
            continue
        for checkpoint_group in ('base', 'final'):
            stats_path = try_stats_path_for_group(eval_owl_dir, checkpoint_group)
            if stats_path is None:
                continue
            try:
                stats = load_json(stats_path)
            except Exception as e:
                # skip unreadable files
                continue
            # extract scalar numeric top-level keys and simple nested numeric fields
            def extract(prefix: str, value):
                rows = []
                if isinstance(value, (int, float)) and not isinstance(value, bool):
                    rows.append((prefix, float(value)))
                elif isinstance(value, dict):
                    for k, v in value.items():
                        if isinstance(v, (int, float)) and not isinstance(v, bool):
                            rows.append((f'{prefix}.{k}', float(v)))
                return rows
            for key, val in stats.items():
                for metric_name, metric_value in extract(key, val):
                    metrics_rows.append({
                        'hop': hop,
                        'run_key': run_key,
                        'run_label': RUNS[run_key]['short_label'],
                        'dataset_dir': RUNS[run_key]['label'],
                        'checkpoint_group': checkpoint_group,
                        'stats_path': str(stats_path),
                        'metric_name': metric_name,
                        'metric_value': metric_value,
                    })

metrics_df = pd.DataFrame(metrics_rows)
metrics_path = CSV_DIR / 'stats_metrics_summary.csv'
if not metrics_df.empty:
    metrics_df.to_csv(metrics_path, index=False)
metrics_df.head()

,hop,run_key,run_label,dataset_dir,checkpoint_group,stats_path,metric_name,metric_value
0,1,lora,lora-8,filtered-dataset-lora-8-seed-42,base,/home/abasso_aims_ac_za/divergence-tokens/work...,mean,0.01120
1,1,lora,lora-8,filtered-dataset-lora-8-seed-42,base,/home/abasso_aims_ac_za/divergence-tokens/work...,margin_error,0.00693
2,1,lora,lora-8,filtered-dataset-lora-8-seed-42,base,/home/abasso_aims_ac_za/divergence-tokens/work...,lower_bound,0.00427
3,1,lora,lora-8,filtered-dataset-lora-8-seed-42,base,/home/abasso_aims_ac_za/divergence-tokens/work...,upper_bound,0.01813
4,1,lora,lora-8,filtered-dataset-lora-8-seed-42,base,/home/abasso_aims_ac_za/divergence-tokens/work...,count,50.00000


In [11]:
# Plot stats.json metrics with error bars, all combinations on one plot
# Group by metric and plot all checkpoint groups and runs together

import numpy as np

if metrics_df.empty:
    print('No stats.json metrics to plot.')
else:
    # Extract the base metric name (without mean/std/error suffixes)
    metrics_df_plot = metrics_df.copy()
    
    # Focus on key metrics that have multiple checkpoint groups and runs
    # Group by metric_name and create plots for top metrics
    unique_metrics = sorted(metrics_df_plot['metric_name'].unique())
    
    # Filter to just the main metrics (mean, margin_error, etc. - skip count, bounds for now)
    key_metrics = [m for m in unique_metrics if m in ['mean', 'margin_error', 'lower_bound', 'upper_bound']]
    
    if not key_metrics:
        key_metrics = unique_metrics[:6]  # fallback to first 6 metrics
    
    n_metrics_per_row = 2
    n_rows = (len(key_metrics) + n_metrics_per_row - 1) // n_metrics_per_row
    
    fig, axes = plt.subplots(n_rows, n_metrics_per_row, figsize=(14, 5*n_rows))
    axes_flat = np.atleast_1d(axes).flatten()
    
    # Color scheme for different runs and checkpoint groups
    colors = {
        ('lora', 'base'): '#1f77b4',
        ('lora', 'final'): '#1f77b4',
        ('dpoints_only', 'base'): '#ff7f0e',
        ('dpoints_only', 'final'): '#ff7f0e',
    }
    
    linestyles = {
        'base': '-',
        'final': '--',
    }
    
    markers = {
        'base': 'o',
        'final': 's',
    }
    
    for ax_idx, metric_name in enumerate(key_metrics):
        if ax_idx >= len(axes_flat):
            break
        ax = axes_flat[ax_idx]
        
        metric_data = metrics_df_plot[metrics_df_plot['metric_name'] == metric_name].copy()
        
        if metric_data.empty:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            continue
        
        # Plot lines for each combination of run_key and checkpoint_group
        for run_key in ['lora', 'dpoints_only']:
            for checkpoint_group in ['base', 'final']:
                subset = metric_data[
                    (metric_data['run_key'] == run_key) & 
                    (metric_data['checkpoint_group'] == checkpoint_group)
                ].sort_values('hop')
                
                if subset.empty:
                    continue
                
                # Prepare error bars using margin_error if available, otherwise use std
                error_values = None
                if 'margin_error' in metrics_df_plot['metric_name'].values:
                    # Try to get corresponding margin_error for this subset
                    error_subset = metric_data[
                        (metric_data['metric_name'] == 'margin_error') &
                        (metric_data['run_key'] == run_key) & 
                        (metric_data['checkpoint_group'] == checkpoint_group)
                    ].sort_values('hop')
                    if not error_subset.empty:
                        error_values = error_subset['metric_value'].values
                
                # Create label for legend
                run_label = RUNS[run_key]['short_label']
                ckpt_label = 'Ckpt' if checkpoint_group == 'final' else 'Base'
                label = f'{run_label} {ckpt_label}'
                
                # Plot with error bars
                ax.errorbar(
                    subset['hop'],
                    subset['metric_value'],
                    yerr=error_values if error_values is not None else 0,
                    fmt=markers[checkpoint_group],
                    linestyle=linestyles[checkpoint_group],
                    capsize=4,
                    capthick=1.5,
                    linewidth=2,
                    markersize=6,
                    label=label,
                    color=colors[(run_key, checkpoint_group)],
                    ecolor=colors[(run_key, checkpoint_group)],
                    elinewidth=1.5,
                    alpha=0.8,
                )
        
        ax.set_title(f'Eval-OWL {metric_name} by hop', fontsize=11, fontweight='bold')
        ax.set_xlabel('Hop')
        ax.set_ylabel(metric_name)
        ax.set_xticks(sorted(metrics_df_plot['hop'].unique()))
        ax.grid(True, alpha=0.3, linestyle=':')
        ax.legend(loc='best', fontsize=9)
    
    # Hide unused subplots
    for ax in axes_flat[len(key_metrics):]:
        ax.set_visible(False)
    
    plot_filename = 'stats_metrics_all_runs_and_checkpoints.png'
    plot_path = save_figure(fig, plot_filename)
    print(f"Saved: {plot_path}")

print("\nDone plotting stats.json metrics (all runs and checkpoints together).")

Saved: /home/abasso_aims_ac_za/divergence-tokens/notebooks/hop_logprob_analysis/images/stats_metrics_all_runs_and_checkpoints.png

Done plotting stats.json metrics (all runs and checkpoints together).


## Metrics summary and reference

This cell shows a complete reference of all metrics discovered in the stats.json files across hops and runs. The code below writes a metrics reference file for future inspection.


In [12]:
# Generate metrics reference file and summary table
if not metrics_df.empty:
    # Create a summary of all metrics by hop and run
    metrics_summary = metrics_df[metrics_df['checkpoint_group'] == 'final'].groupby('metric_name').agg({
        'metric_value': ['min', 'max', 'mean', 'std'],
        'hop': 'nunique',
        'run_key': 'nunique',
    }).round(6)
    
    metrics_summary.columns = ['_'.join(col).strip() for col in metrics_summary.columns]
    metrics_summary = metrics_summary.reset_index().sort_values('metric_name')
    
    # Write to CSV for reference
    metrics_reference_path = CSV_DIR / 'metrics_reference_summary.csv'
    metrics_summary.to_csv(metrics_reference_path, index=False)
    
    # Also write a human-readable text file
    metrics_txt_path = CSV_DIR / 'metrics_reference.txt'
    with open(metrics_txt_path, 'w') as f:
        f.write("=== METRICS DISCOVERED IN STATS.JSON ===\n\n")
        f.write(f"Total unique metrics: {len(metrics_summary)}\n")
        f.write(f"Hops covered: {sorted(metrics_df['hop'].unique())}\n")
        f.write(f"Runs: {', '.join(RUNS[k]['short_label'] for k in RUNS)}\n\n")
        f.write("Metrics by name:\n")
        f.write("-" * 80 + "\n")
        for idx, row in metrics_summary.iterrows():
            f.write(f"\n{row['metric_name']}\n")
            f.write(f"  Range: [{row['metric_value_min']:.6f}, {row['metric_value_max']:.6f}]\n")
            f.write(f"  Mean ± Std: {row['metric_value_mean']:.6f} ± {row['metric_value_std']:.6f}\n")
            f.write(f"  Hops: {int(row['hop_nunique'])}, Runs: {int(row['run_key_nunique'])}\n")
    
    print(f"Metrics reference written to {metrics_reference_path} and {metrics_txt_path}")
    print("\nMetrics Summary:")
    print(metrics_summary)
else:
    print("No metrics found to summarize.")

Metrics reference written to /home/abasso_aims_ac_za/divergence-tokens/notebooks/hop_logprob_analysis/csv/metrics_reference_summary.csv and /home/abasso_aims_ac_za/divergence-tokens/notebooks/hop_logprob_analysis/csv/metrics_reference.txt

Metrics Summary:
    metric_name  metric_value_min  metric_value_max  metric_value_mean  \
0    confidence          0.950000          0.950000           0.950000   
1         count         50.000000         50.000000          50.000000   
2   lower_bound          0.006489          0.062239           0.025599   
3  margin_error          0.009033          0.041357           0.021396   
4          mean          0.017700          0.101400           0.046994   
5   upper_bound          0.028133          0.142757           0.068390   

   metric_value_std  hop_nunique  run_key_nunique  
0          0.000000            9                2  
1          0.000000            9                2  
2          0.016770            9                2  
3          0.010

## Complete output reference

### CSV files (for replotting and analysis):
- `csv/hop_checkpoint_summary_long.csv` — Hop averages (base and final checkpoints)
- `csv/hop_checkpoint_summary_wide.csv` — Base vs final comparison table
- `csv/question_logprob_long.csv` — Per-question logprob values
- `csv/question_delta_within_run_long.csv` — Per-question improvements
- `csv/question_final_comparison_wide.csv` — Per-question run comparison
- `csv/stats_metrics_summary.csv` — All discovered stats.json metrics (long format)
- `csv/metrics_reference_summary.csv` — Statistical summary of all metrics (min/max/mean/std)
- `csv/metrics_reference.txt` — Human-readable metrics reference guide

### Plots (saved as PNG):

**Logprob and probability analysis:**
- `images/final_checkpoint_mean_logprob_and_p_by_hop.png` — Main plot (log p and p)
- `images/final_checkpoint_gap_by_hop.png` — Run comparison gap
- `images/within_run_final_minus_base_change_by_hop.png` — Training effect by run
- `images/paired_final_checkpoint_scatter_by_question.png` — Question-level run scatter

**Per-question diagnostics:**
- `images/question_delta_heatmap_by_run.png` — Per-question improvements heatmap
- `images/question_delta_boxplot_by_run.png` — Per-question improvement distribution

**Stats.json metrics with error bars (separated by run):**
- `images/stats_metrics_by_hop_lora-8.png` — All stats.json metrics (lora-8 run, mean ± std by hop)
- `images/stats_metrics_by_hop_dpoints-only.png` — All stats.json metrics (dpoints-only run, mean ± std by hop)